### Robustness Analysis

The primary analysis estimated a descriptive fixed-effects panel regression of annual changes in unemployment on labour market coordination and macroeconomic controls.

The main result was that COORD had a negative but statistically insignificant coefficient. This suggests weak evidence that higher labour market coordination is associated with smaller increases in unemployment once controls and fixed effects are included.

In [12]:
import pandas as pd

# Load main panel
panel = pd.read_csv("data/clean/final_panel.csv")

# Load institutional variables
inst = pd.read_csv(
    "data/clean/InstitutionalVariables_clean.csv"
)

# Keep only needed columns
inst = inst[[
    "Country",
    "Year",
    "UnionDensity",
    "BargainingCoverage"
]]

# Merge into panel
panel = panel.merge(
    inst,
    on=["Country", "Year"],
    how="left"
)

# Create regression dataset
reg_data = panel.dropna(subset=[
    "Delta_Unemployment",
    "GDPGrowth",
    "Inflation",
    "ExportDependence",
    "GovSpend"
]).copy()

# Check columns
print(reg_data.columns.tolist())

['Country', 'Year', 'Unemployment', 'Delta_Unemployment', 'COORD', 'GDPGrowth', 'Inflation', 'ExportDependence', 'GovSpend', 'UnionDensity', 'BargainingCoverage']


For this project, the Robustness tests are as follows. This is because this is a **descriptive** model, as such the robustness checks focus primarily on specification sensitivity, omitted-variable concerns, and sample dependence rather than formal causal-identification tests such as IV or DiD.


| Section Number | Section Name | Tests per Section | Purpose |
|---|---|---|---|
| `1` | Main specification | `1` | A summary of original results for easier comparison |
| `2` | No Controls | `1` | Tests whether the significance of COORD is dependent on macroeconomic conditions |
| `3` | Dropping Country FE | `3` | Tests whether COORD only works after controls |
| `4` | Drop Year FE | `1` |  Tests whether COORD only works after controls |
| `5` | Replacing COORD with counterparts | `2` | Tests whether COORD is proxying for union density / bargaining coverage |
| `6` | Dropping Countries | `16` | R-estimates the model 16 times, each time excluding one country per regression |

### 1. Main Specification

In [16]:
baseline = smf.ols(
    """
    Delta_Unemployment ~
    COORD +
    GDPGrowth +
    Inflation +
    ExportDependence +
    GovSpend +
    C(Country) +
    C(Year)
    """,
    data=reg_data
).fit(cov_type="HC1")

### 2. No Controls

In [2]:
no_controls = smf.ols(
    """
    Delta_Unemployment ~ COORD + C(Country) + C(Year)
    """,
    data=reg_data
).fit(cov_type="HC1")

### 3. Dropping Country FE

In [3]:
no_country_fe = smf.ols(
    """
    Delta_Unemployment ~ COORD + GDPGrowth + Inflation + ExportDependence + GovSpend
    + C(Year)
    """,
    data=reg_data
).fit(cov_type="HC1")

### 4. Dropping Year FE

In [4]:
no_year_fe = smf.ols(
    """
    Delta_Unemployment ~ COORD + GDPGrowth + Inflation + ExportDependence + GovSpend
    + C(Country)
    """,
    data=reg_data
).fit(cov_type="HC1")

### 5. Replacing COORD with Counterparts

**Union Density**

In [18]:
union_model = smf.ols(
    """
    Delta_Unemployment ~ UnionDensity + GDPGrowth + Inflation
    + ExportDependence + GovSpend
    + C(Country) + C(Year)
    """,
    data=reg_data
).fit(cov_type="HC1")

**BargainingCoverage**

In [10]:
coverage_model = smf.ols(
    """
    Delta_Unemployment ~
    BargainingCoverage +
    GDPGrowth +
    Inflation +
    ExportDependence +
    GovSpend +
    C(Country) +
    C(Year)
    """,
    data=reg_data
).fit(cov_type="HC1")

print(coverage_model.summary())

                            OLS Regression Results                            
Dep. Variable:     Delta_Unemployment   R-squared:                       0.540
Model:                            OLS   Adj. R-squared:                  0.478
Method:                 Least Squares   F-statistic:                     8.634
Date:                Thu, 14 May 2026   Prob (F-statistic):           2.19e-31
Time:                        20:51:58   Log-Likelihood:                -274.44
No. Observations:                 342   AIC:                             632.9
Df Residuals:                     300   BIC:                             793.9
Df Model:                          41                                         
Covariance Type:                  HC1                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

c:\Users\renee\OneDrive\Desktop\ECC3479\Project\ECC3479_UNITPROJECT\.venv\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 42, but rank is 41
  warnings.warn('covariance of constraints does not have full '


### 6. Leave-One-Country-Out

In [7]:
countries = reg_data["Country"].unique()

leave_out_results = []

for country in countries:

    temp = reg_data[
        reg_data["Country"] != country
    ]

    model = smf.ols(
        """
        Delta_Unemployment ~ COORD + GDPGrowth + Inflation
        + ExportDependence + GovSpend
        + C(Country) + C(Year)
        """,
        data=temp
    ).fit(cov_type="HC1")

    leave_out_results.append({
        "Dropped Country": country,
        "COORD Coef": round(model.params["COORD"], 4),
        "P-value": round(model.pvalues["COORD"], 4)
    })

leave_out_df = pd.DataFrame(leave_out_results)

display(leave_out_df)

,Dropped Country,COORD Coef,P-value
0,Australia,-0.0183,0.9180
1,Austria,-0.0382,0.7932
2,Belgium,-0.0468,0.7456
3,Canada,-0.0608,0.6749
4,Denmark,-0.0594,0.6811
5,Finland,-0.0418,0.8541
6,Germany,-0.0338,0.8200
7,Japan,-0.0609,0.6768
8,Netherlands,-0.0403,0.7688
9,New Zealand,-0.0583,0.6840


### Robustness Summary Table

In [20]:
# --------------------------------------------------
# Robustness Summary Table
# Includes main robustness checks + leave-one-country-out
# --------------------------------------------------

models = {
    "Baseline": baseline,
    "No Controls": no_controls,
    "No Country FE": no_country_fe,
    "No Year FE": no_year_fe,
    "Union Density": union_model,
    "Bargaining Coverage": coverage_model
}

rows = []

# Main robustness models
for name, model in models.items():

    variable = (
        "COORD"
        if "COORD" in model.params.index
        else "UnionDensity"
        if "UnionDensity" in model.params.index
        else "BargainingCoverage"
    )

    rows.append({
        "Specification": name,
        "Variable": variable,
        "Coefficient": round(model.params[variable], 4),
        "Std Error": round(model.bse[variable], 4),
        "P-value": round(model.pvalues[variable], 4),
        "N": int(model.nobs),
        "R²": round(model.rsquared, 4)
    })

# Leave-one-country-out robustness
leave_out_results = []

for country in reg_data["Country"].unique():

    temp = reg_data[reg_data["Country"] != country].copy()

    loo_model = smf.ols(
        """
        Delta_Unemployment ~
        COORD +
        GDPGrowth +
        Inflation +
        ExportDependence +
        GovSpend +
        C(Country) +
        C(Year)
        """,
        data=temp
    ).fit(cov_type="HC1")

    leave_out_results.append({
        "Dropped Country": country,
        "COORD Coefficient": round(loo_model.params["COORD"], 4),
        "Std Error": round(loo_model.bse["COORD"], 4),
        "P-value": round(loo_model.pvalues["COORD"], 4),
        "N": int(loo_model.nobs),
        "R²": round(loo_model.rsquared, 4)
    })

leave_out_df = pd.DataFrame(leave_out_results)

# Convert leave-one-country-out into summary rows
for _, row in leave_out_df.iterrows():
    rows.append({
        "Specification": f"Drop {row['Dropped Country']}",
        "Variable": "COORD",
        "Coefficient": row["COORD Coefficient"],
        "Std Error": row["Std Error"],
        "P-value": row["P-value"],
        "N": row["N"],
        "R²": row["R²"]
    })

robustness_table = pd.DataFrame(rows)

display(robustness_table)

# Save outputs
robustness_table.to_csv("data/clean/robustness_table.csv", index=False)
leave_out_df.to_csv("data/clean/leave_one_country_out.csv", index=False)

print("Saved robustness_table.csv and leave_one_country_out.csv")

,Specification,Variable,Coefficient,Std Error,P-value,N,R²
0,Baseline,COORD,-0.0501,0.1442,0.7284,365,0.5220
1,No Controls,COORD,-0.1070,0.1557,0.4920,365,0.4584
2,No Country FE,COORD,-0.0804,0.0380,0.0343,365,0.4876
3,No Year FE,COORD,-0.0039,0.1373,0.9775,365,0.3909
4,Union Density,UnionDensity,-0.0023,0.0143,0.8701,342,0.5369
5,Bargaining Coverage,BargainingCoverage,0.0187,0.0110,0.0891,342,0.5405
6,Drop Australia,COORD,-0.0183,0.1781,0.9180,342,0.5382
7,Drop Austria,COORD,-0.0382,0.1458,0.7932,342,0.5325
8,Drop Belgium,COORD,-0.0468,0.1443,0.7456,342,0.5320
9,Drop Canada,COORD,-0.0608,0.1451,0.6749,342,0.5102


Saved robustness_table.csv and leave_one_country_out.csv


### Robustness Interpretation


| Base-line Result          | Interpretation                                                                 |
| --------------- | ------------------------------------------------------------------------------ |
| COORD = -0.0501 | Higher coordination is associated with slightly smaller unemployment increases |
| p = 0.7284      | Not statistically significant                                                  |
| R² = 0.522      | Model explains ~52% of unemployment variation                                  |


After controlling for macroeconomic variables and fixed effects, higher labour market coordination is associated with slightly lower unemployment changes, although the relationship is statistically insignificant. 

| Result          | Interpretation               |
| --------------- | ---------------------------- |
| COORD = -0.1070 | Larger negative relationship |
| p = 0.492       | Still insignificant          |

Part of the observed relationship between COORD and unemployment operates through macroeconomic conditions such as GDP growth and inflation. 

| Result          | Interpretation                  |
| --------------- | ------------------------------- |
| COORD = -0.0804 | Stronger effect                 |
| p = 0.0343      | Statistically significant at 5% |

Once country fixed effects are removed, labour market coordination becomes statistically significant and negatively associated with unemployment changes. As stated within the Primary Econometric Analysis, country fixed effects absorb much of the long-run institutional variation because labour market coordination varies mostly across countries rather than within countries over time. 

| Result    | Interpretation           |
| --------- | ------------------------ |
| COORD ≈ 0 | Almost no relationship   |
| p = 0.978 | Completely insignificant |

The relationship between labour market institutions and unemployment resilience is highly sensitive to common global macroeconomic shocks.

| Result                | Interpretation               |
| --------------------- | ---------------------------- |
| coefficient = -0.0023 | Slight negative relationship |
| p = 0.870             | Very insignificant           |

Union density alone does not appear strongly associated with unemployment resilience once controls and FE are included.

| Result               | Interpretation                |
| -------------------- | ----------------------------- |
| coefficient = 0.0187 | Slight positive relationship  |
| p = 0.0891           | Marginally significant (~10%) |


Higher bargaining coverage may be associated with slightly larger unemployment increases, although the evidence is weak and sensitive to specification. The sign differs from the COORD specification, suggesting that different institutional measures may capture different labour market mechanisms. An interesting explanation for the positive bargaining coverage coefficient may relate to broader labour-market rigidities associated with highly extended collective bargaining systems. Murtin et al. (2014), in “Unemployment and the coverage extension of collective wage agreements”, identify a significant positive relationship between tax wedges and the extension of collective bargaining coverage. The authors further argue that tax wedges may have stronger positive effects on unemployment in countries with broad collective bargaining extension systems, such as France and Spain. 

The theorethical idea can be explained simply: COORD can improve coordinated adjustment --> but bargaining coverage extension can sometimes increase financial rigidity for employers especially when combined with high labour taxation (tax wedge) --> hiring more expensive during downturns, and therefore increasing unemployment. Though, the positive bargaining coverage coefficient should not be interpreted causally. Rather, it may reflect broader institutional rigidities associated with highly extended bargaining systems.

Lastly, every result from the Leave-one-country out shows the same trend: negative, small, insignificant, stable; hence, the estimated relationship is not driven disproportionately by any individual country within the sample.

Overall, despite the explained Collective Bargaining Coverage difference in results, all other tests have held to tests that the countries with stronger coordinated labour-market institutions tended to experience different lower unermployment outcomes, with the relationship remaining negative throughout the variety of robustness tests.


### Conclusion

The robustness checks suggest that the estimated relationship between labour market coordination and unemployment resilience is generally stable across alternative specifications and country samples.

The strongest result emerges when country fixed effects are removed, where COORD becomes statistically significant and negatively associated with unemployment changes. This suggests that country fixed effects absorb much of the institutional variation because labour market coordination changes slowly within countries over time.

Alternative institutional measures such as union density resulted similarly alike COORD (ie. positive and insignificant with controls), whereas Wage Coordination resulted in a positive result which can be explained through other observed relationships studied in the macroeconomic field (ie. the interaction between positive relationship between wage coordination and tax wedge, and therefore the positive correlation with unemployment). The leave-one-country-out analysis demonstrates that the overall findings are not driven by any single country in the sample.